# 03 - MACE Training
Train the message-passing MACE model using exactly the same hyperparameters.


In [1]:
import sys
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import DataLoader
import pandas as pd

sys.path.append("..")
from src.dataset import MaterialsProjectDataset
from src.models.mace_wrapper import MACEWrapper
from src.trainer import BenchmarkTrainer


In [2]:
# Load Data
train_ds = MaterialsProjectDataset("../data/train.extxyz", cutoff=5.0)
val_ds = MaterialsProjectDataset("../data/val.extxyz", cutoff=5.0)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)


Pre-computing graphs for 1000 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/1000 [00:00<?, ?it/s]

  Done. Dataset ready (1000 graphs).
Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).


In [3]:
# Initialize Model with identical hyperparameters
model = MACEWrapper(
    num_elements=120,
    r_max=5.0,
    num_radial=8,
    l_max=2,
    num_blocks=2, # 2 layers of message passing
    node_dim=16
)

optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

trainer = BenchmarkTrainer(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    device="cuda" if torch.cuda.is_available() else "cpu",
    energy_weight=1.0,
    force_weight=100.0
)


C:\Users\Prabhat\AppData\Local\Programs\Python\Python314\Lib\ast.py:506: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
C:\Users\Prabhat\AppData\Local\Programs\Python\Python314\Lib\ast.py:506: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
C:\Users\Prabhat\AppData\Local\Programs\Python\Python314\Lib\ast.py:506: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
C:\Users\Prabhat\AppData\

In [4]:
# Train
metrics_df = trainer.train(max_epochs=50, patience=10)
metrics_df.to_csv("../data/mace_metrics.csv", index=False)
metrics_df.head()


Epoch 000 | Time: 400.67s | Train E MAE: 14778.64 meV/atom | Train F MAE: 208.50 meV/Å | Val E MAE: 15010.65 meV/atom | Val F MAE: 353.38 meV/Å
Epoch 001 | Time: 387.56s | Train E MAE: 14183.00 meV/atom | Train F MAE: 918.80 meV/Å | Val E MAE: 14528.95 meV/atom | Val F MAE: 1273.13 meV/Å
Epoch 002 | Time: 384.51s | Train E MAE: 13441.84 meV/atom | Train F MAE: 1447.11 meV/Å | Val E MAE: 14093.10 meV/atom | Val F MAE: 821.44 meV/Å
Epoch 003 | Time: 389.61s | Train E MAE: 12739.84 meV/atom | Train F MAE: 1222.39 meV/Å | Val E MAE: 13599.96 meV/atom | Val F MAE: 1345.68 meV/Å
Epoch 004 | Time: 386.08s | Train E MAE: 12883.82 meV/atom | Train F MAE: 1242.04 meV/Å | Val E MAE: 13387.14 meV/atom | Val F MAE: 1151.34 meV/Å
Epoch 005 | Time: 384.60s | Train E MAE: 12706.42 meV/atom | Train F MAE: 1251.16 meV/Å | Val E MAE: 13622.48 meV/atom | Val F MAE: 1072.81 meV/Å
Epoch 006 | Time: 387.46s | Train E MAE: 12382.17 meV/atom | Train F MAE: 1247.41 meV/Å | Val E MAE: 13241.94 meV/atom | Val F M

KeyboardInterrupt: 